# 12. Blockwise distances and ranking

![Blockwise ranking](../images/12_blockwise_distances_and_ranking.svg)

**Learning goals:** compute cosine distances in batches, aggregate uneven query blocks with equal-factor weights, preserve ties, and calculate fractional top-1, average tied rank, and mean reciprocal rank.

In [ ]:
from itertools import product
import numpy as np
import matplotlib.pyplot as plt

SEED = 12
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Pairwise cosine distance

For nonzero vectors $x$ and $y$, cosine distance is $1-x^Ty/(\lVert x\rVert_2\lVert y\rVert_2)$. Normalize every row once, then multiply a query matrix of shape `(n_queries, p)` by a gallery transpose of shape `(p, n_gallery)`. This replaces two Python loops with optimized matrix multiplication. A zero vector has no direction, so this helper rejects one explicitly.

In [ ]:
def normalize_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    if np.any(norms <= eps):
        raise ValueError("cosine distance is undefined for a zero vector")
    return x / norms

def cosine_distances(queries, gallery):
    similarity = normalize_rows(queries) @ normalize_rows(gallery).T
    return 1.0 - np.clip(similarity, -1.0, 1.0)

gallery = np.array([[1., 0.], [0., 1.], [-1., 0.]])
queries = np.array([[1., 1.], [1., 0.]])
D = cosine_distances(queries, gallery)
assert D.shape == (2, 3) and np.all(D >= 0)
print("pairwise distance matrix:\n", D)

## 2. Nested aggregation gives each factor one vote

We now simulate distances from uneven factor-cell blocks to four gallery items. Both factor and cell change the expected distance. The scientific estimand first averages observations within each cell, gives cells equal weight inside each factor, then gives factors equal weight. A global row mean overweights the large cell in factor 0. The implementation keeps all gallery columns together, so each group operation returns one vector of length `n_gallery`.

In [ ]:
factor = np.array([0] * 8 + [1] * 4)
cell = np.array([0] * 6 + [1] * 2 + [0] * 2 + [1] * 2)
base_by_factor_cell = np.array([
    [[.10, .20, .70, .85], [.50, .60, .40, .55]],  # factor 0, cells 0 and 1
    [[.65, .45, .25, .65], [.85, .55, .15, .45]],  # factor 1, cells 0 and 1
])
distances = np.vstack([base_by_factor_cell[f, c] + rng.normal(0, .01, 4)
                       for f, c in zip(factor, cell)])

def nested_equal_means(values, factors, cells):
    factor_means = []
    for f in np.unique(factors):
        in_factor = factors == f
        cell_means = [values[in_factor & (cells == c)].mean(axis=0)
                      for c in np.unique(cells[in_factor])]
        factor_means.append(np.mean(cell_means, axis=0))
    factor_means = np.stack(factor_means)
    return factor_means, factor_means.mean(axis=0)

factor_equal_cell, structured = nested_equal_means(distances, factor, cell)
global_mean = distances.mean(axis=0)
factor0_row_weighted = distances[factor == 0].mean(axis=0)
assert factor_equal_cell.shape == (2, 4) and structured.shape == (4,)
assert not np.allclose(factor0_row_weighted, factor_equal_cell[0], atol=.02)
assert not np.allclose(global_mean, structured, atol=.02)
assert not np.allclose(factor_equal_cell[0], factor_equal_cell[1], atol=.02)
print("global row-weighted:             ", global_mean.round(3))
print("factor 0, equal cells within:    ", factor_equal_cell[0].round(3))
print("factor 1, equal cells within:    ", factor_equal_cell[1].round(3))
print("final equal-factor result:       ", structured.round(3))

## 3. Ties need metric-level treatment

Sorting gives tied items an arbitrary display order. Instead, define each tie set against one reference using a documented tolerance. Fractional top-1 divides credit uniformly among tied leaders. Average tied rank gives every member of a tie the mean of the positions occupied by that tie.

In [ ]:
def fractional_top1(scores, positive_index, atol=1e-12):
    scores = np.asarray(scores)
    top = np.isclose(scores, scores.min(), rtol=0.0, atol=atol)
    return float(top[positive_index]) / int(top.sum())

def average_tied_rank(scores, positive_index, atol=1e-12):
    scores = np.asarray(scores, dtype=np.float64)
    reference = scores[positive_index]
    strictly_better = int(np.sum(scores < reference - atol))
    tied = int(np.sum(np.abs(scores - reference) <= atol))
    return strictly_better + (tied + 1) / 2

tied_scores = np.array([.10, .10, .40, .80])
credit = fractional_top1(tied_scores, positive_index=1)
rank = average_tied_rank(tied_scores, positive_index=1)
assert credit == .5 and rank == 1.5
print(f"fractional top-1={credit:.2f}; average tied rank={rank}")

## 4. Mean reciprocal rank across queries

For query $i$, reciprocal rank is $1/r_i$. MRR is their arithmetic mean. It strongly rewards moving a positive near the top. Here we include a top tie, a rank-2 result, and a rank-4 result so the calculation can be checked by hand.

In [ ]:
score_rows = np.array([[.1, .1, .4, .8],
                       [.3, .1, .5, .7],
                       [.2, .3, .4, .5]])
positives = np.array([1, 0, 3])
ranks = np.array([average_tied_rank(row, pos) for row, pos in zip(score_rows, positives)])
credits = np.array([fractional_top1(row, pos) for row, pos in zip(score_rows, positives)])
mrr = np.mean(1.0 / ranks)
assert ranks.tolist() == [1.5, 2, 4]
assert np.isclose(mrr, (2 / 3 + .5 + .25) / 3)
print("ranks:", ranks.tolist())
print(f"mean fractional top-1={credits.mean():.3f}; MRR={mrr:.3f}")

## 5. Inspect a ranking without erasing ties

The plot orders gallery items by score for readability and colors the positive. The metric still uses original scores and an explicit tolerance. For a very large gallery, `np.argpartition` finds top-k candidates without fully sorting, although it does not order those k candidates.

In [ ]:
scores = tied_scores
positive = 1
order = np.argsort(scores, kind="stable")
colors = ["#d89b21" if idx == positive else "#69a7d0" for idx in order]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(np.arange(len(order)), scores[order], color=colors)
ax.set(xticks=np.arange(len(order)), xticklabels=[f"item {i}" for i in order],
       ylabel="distance (lower is better)", title="A top tie shares credit")
ax.set_ylim(0, .9)
plt.tight_layout()
plt.show()

## 6. End-to-end synthetic GFC-v2 retrieval

This evaluator uses three donor-supplied factor blocks and all eight gallery recordings. It runs zero-, one-, two-, and three-recovered-factor oracles through the same query and ranking code. Opposite directions at fixed speed and clothing share a synthetic source ID, so a direction-focal query must fail the source-lineage gate. The allowed speed- and clothing-focal queries keep both donors in the gallery, average one cosine distance per factor, and use one tolerance for both tie-aware metrics.

In [ ]:
factor_names = ("speed", "clothing", "direction")
gfc_cells = list(product([0, 1], repeat=3))
level_vectors = {0: np.array([1.0, 0.0]), 1: np.array([0.0, 1.0])}

def build_oracle_recordings(recovered_factors):
    recovered_factors = set(recovered_factors)
    return {
        cell: {"source_video_id": f"source-speed{cell[0]}-clothing{cell[1]}",
               "blocks": {name: (level_vectors[cell[j]].copy()
                                  if name in recovered_factors
                                  else np.array([1.0, 0.0]))
                          for j, name in enumerate(factor_names)}}
        for cell in gfc_cells
    }

def complementary_donors(target, focal_index):
    donor_u = tuple(value if j == focal_index else 1 - value
                    for j, value in enumerate(target))
    donor_v = tuple(1 - value if j == focal_index else value
                    for j, value in enumerate(target))
    return donor_u, donor_v

def require_source_separation(recordings, target, donor_u, donor_v):
    target_source = recordings[target]["source_video_id"]
    if any(recordings[donor]["source_video_id"] == target_source
           for donor in (donor_u, donor_v)):
        raise ValueError("donor and target source_video_id must differ")

def gfc_query_metrics(recordings, target, focal_index, tolerance=1e-12):
    donor_u, donor_v = complementary_donors(target, focal_index)
    require_source_separation(recordings, target, donor_u, donor_v)
    query_blocks = {}
    for j, name in enumerate(factor_names):
        source = donor_u if j == focal_index else donor_v
        query_blocks[name] = recordings[source]["blocks"][name]

    candidate_distances = []
    for candidate in gfc_cells:  # complete gallery, including both donors
        per_factor = [cosine_distances(query_blocks[name][None, :],
                                        recordings[candidate]["blocks"][name][None, :])[0, 0]
                      for name in factor_names]
        candidate_distances.append(np.mean(per_factor, dtype=np.float64))
    candidate_distances = np.asarray(candidate_distances, dtype=np.float64)
    target_index = gfc_cells.index(target)
    return (fractional_top1(candidate_distances, target_index, atol=tolerance),
            average_tied_rank(candidate_distances, target_index, atol=tolerance),
            donor_u, donor_v)

oracle_results = {}
for recovered_count in range(4):
    recovered = factor_names[3 - recovered_count:]
    recordings = build_oracle_recordings(recovered)
    results = [gfc_query_metrics(recordings, target, focal_index)
               for target in gfc_cells for focal_index in (0, 1)]
    assert len(results) == 16
    assert all(u in gfc_cells and v in gfc_cells for _, _, u, v in results)
    oracle_results[recovered_count] = {
        "top1": np.mean([result[0] for result in results]),
        "mrr": np.mean([1.0 / result[1] for result in results]),
    }

expected_top1 = {0: 1 / 8, 1: 1 / 4, 2: 1 / 2, 3: 1.0}
expected_mrr = {count: 2 / (2 ** (3 - count) + 1) for count in range(4)}
assert all(np.isclose(oracle_results[count]["top1"], expected_top1[count])
           for count in range(4))
assert all(np.isclose(oracle_results[count]["mrr"], expected_mrr[count])
           for count in range(4))

shared_source_recordings = build_oracle_recordings(factor_names)
direction_rejections = 0
for target in gfc_cells:
    try:
        require_source_separation(shared_source_recordings, target,
                                  *complementary_donors(target, 2))
    except ValueError:
        direction_rejections += 1
assert direction_rejections == 8
print({count: {name: round(value, 3) for name, value in metrics.items()}
       for count, metrics in oracle_results.items()})
print("direction-focal source violations rejected:", direction_rejections)

## Exercises and takeaways

1. Change the top scores to `[0.1, 0.1, 0.1, 0.8]` with item 1 positive. Predict both metrics.
2. Give factor 0 one hundred rows and factor 1 two rows. Compare global and equal-factor means.
3. Add a zero query vector and explain why the helper raises an error.

**Brief answers:** the positive receives fractional top-1 $1/3$ and average tied rank 2. Global averaging approaches factor 0's mean, while equal-factor averaging still gives each factor one half of the weight. A zero vector has zero magnitude and no defined direction, so cosine distance is undefined.

**Takeaway:** vectorized cosine distance is efficient, but aggregation and tie rules define the scientific meaning of a ranking metric.

## Continue learning

[Previous notebook: 11](11_factorial_state_spaces.ipynb) | [Lecture](../lectures/12_blockwise_distances_and_ranking.md) | [Curriculum](../README.md) | [Next notebook: 13](13_context_interventions.ipynb)